# Embedding flavors

`JointCDRMLM` produces five distinct embeddings for one antibody-antigen pair. This notebook computes all of them for one real pair and shows how they differ.

In [ ]:
import torch
import langaai

model = langaai.load()

In [ ]:
heavy = "EVQLVESGGGLVQPGGSLRLSCAASGFNFKDTYIHWVRQAPGKGLEWVARIYPANGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCASDGGSYSYAFDYWGQGTLVTVSS"
light = "DIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQHYTTPPTFGQGTKVEIK"
antigen = "TQVCTGTDMKLRLPASPETHLDMLRHLYQGCQVVQGNLELTYLPTNASLSFLQDIQEVQGYVLIAHNQVRQVPLQRLRIVRGTQLFEDNYALAVLDNGDPLNNTTPVTGASPGGLRELQLRSLTEILKGGVLIQRNPQLCYQDTILWKDIFHKNNQLALTLIDTNRSRACHPCSPMCKGSRCWGESSEDCQSLTRTVCAGGCARCKGPLPTDCCHEQCAAGCTGPKHSDCLACLHFNHSGICELHCPALVTYNTDTFESMPNPEGRYTFGASCVTACPYNYLSTDVGSCTLVCPLHNQEVTAEDGTQRCEKCSKPCARVCYGLGMEHLREVRAVTSANIQEFAGCKKIFGSLAFLPESFDGDPASNTAPLQPEQLQVFETLEEITGYLYISAWPDSLPDLSVFQNLQVIRGRILHNGAYSLTLQGLGISWLGLRSLRELGSGLALIHHNTHLCFVHTVPWDQLFRNPHQALLHTANRPEDECVGEGLACHQLCARGHCWGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYVNARHCLPCHPECQPQNGSVTCFGPEADQCVACAHYKDPPFCVARCPSGVKPDLSYMPIWKFPDEEGACQPCPIN"

ab = model.encode_antibody(heavy, light)
ag = model.embed_antigen(antigen)

**Antibody**, antigen-blind vs. antigen-conditioned: the blind flavor is plain frozen ESM-2 over the antibody alone (never sees the antigen); the conditioned flavor is post-joint-stack, so it can in principle depend on which antigen was paired with it.

In [ ]:
[ab_blind] = model.antibody_embedding_blind([ab])
[ab_cond] = model.antibody_embedding_conditioned([(ab, ag)])
print("antibody blind:", ab_blind.shape, " conditioned:", ab_cond.shape)
print("blind == conditioned:", torch.equal(ab_blind, ab_cond))

**Antigen**, antibody-blind vs. antibody-conditioned ("paratope-aware"): the blind flavor is exactly `ag.embedding`, the raw input; the conditioned flavor is post-joint-stack.

In [ ]:
[ag_blind] = model.antigen_embedding_blind([ag])
[ag_cond] = model.antigen_embedding_conditioned([(ab, ag)])
print("antigen blind:", ag_blind.shape, " conditioned:", ag_cond.shape)
print("blind == ag.embedding exactly:", torch.equal(ag_blind, ag.embedding))

**cls_embedding**: the one native pair-level pooled vector -- what a downstream regressor (e.g. an affinity predictor fitted separately) would typically be built on.

In [ ]:
[cls] = model.cls_embedding([(ab, ag)])
print("cls:", cls.shape)

pooled_ab = langaai.mean_pool(ab_cond)
pooled_ag = langaai.mean_pool(ag_cond)
print("mean-pooled antibody:", pooled_ab.shape, " mean-pooled antigen:", pooled_ag.shape)

Swap in a different antigen and confirm the antigen-conditioned antibody embedding actually changes.

In [ ]:
other_antigen = model.embed_antigen(light)  # any different sequence, just for contrast
[ab_cond_other] = model.antibody_embedding_conditioned([(ab, other_antigen)])
cos_sim = torch.nn.functional.cosine_similarity(
    langaai.mean_pool(ab_cond).unsqueeze(0), langaai.mean_pool(ab_cond_other).unsqueeze(0)
)
print(f"cosine similarity between the two antigen-conditioned embeddings: {cos_sim.item():.3f}")